<a href="https://colab.research.google.com/github/wiz124/chem169-git/blob/main/Li_Harry_RID_020_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#Exercise 0
mtb_embeddings='otnhfK06zM'
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

def count_amino_acids(sequence):
    amino_acids = list('ACDEFGHIKLMNPQRSTVWY')
    sequence=str(sequence)
    return {aa: sequence.count(aa) for aa in amino_acids}

df=pd.read_excel('mtb_with_localization.xlsx')

location=[]
for entry in df['Subcellular location [CC]']:
  if 'membrane' in str(entry).lower():
    location.append('membrane')
  elif 'secreted' in str(entry).lower():
    location.append('secreted')
  elif 'cytoplasm' in str(entry).lower():
    location.append(  'cytoplasm')
  else:
    location.append(None)

df['location']=location
df.dropna(subset=['location'], inplace=True)

y = df[['location']].to_numpy()

amino_acids = list('ACDEFGHIKLMNPQRSTVWY')

aa_counts = pd.DataFrame(
    df['Sequence'].apply(count_amino_acids).tolist(),
    columns=amino_acids,
    index=df.index
)
df = pd.concat([df, aa_counts], axis=1)


x = df[['Length'] + amino_acids].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,      # 30% for testing
    random_state=42,    # for reproducibility
    stratify=y          # maintain class distribution
)

print(X_train.shape)
print(y_train.shape)

import h5py
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_array
from scipy.sparse.csgraph import connected_components
from scipy.sparse import csr_matrix
from sklearn.model_selection import GroupShuffleSplit


def computesimilarity(embeddings):
  similarity_matrix = np.array(embeddings)
  comp=cosine_similarity(embeddings,embeddings)
  return comp
def clustered_train_test_split(cluster_labels, test_size=0.2, random_state=42):
    unique_clusters = np.unique(cluster_labels)

    train_clusters, test_clusters = train_test_split(
        unique_clusters,
        test_size=test_size,
        random_state=random_state
    )

    train_idx = np.where(np.isin(cluster_labels, train_clusters))[0]
    test_idx  = np.where(np.isin(cluster_labels, test_clusters))[0]

    return train_idx, test_idx

filename=mtb_embeddings

embeddings={}
with h5py.File(filename, 'r') as f:
    for key in list(f.keys()):
        embeddings[key]=f[key][:]

filtered_embeddings=[]
filtered_df=df
for entry in filtered_df['Entry']:
  if entry in embeddings:
    filtered_embeddings.append(embeddings[entry])


filtered_df['embeddings']=filtered_embeddings
filtered_df.dropna(subset=['embeddings'],inplace=True)

for idx, entry in enumerate(filtered_df['Entry']):
  if embeddings[entry].all()==filtered_df['embeddings'].iloc[idx].all():
    continue
  else:
    print('error')

sim_matrix=cosine_similarity(filtered_df['embeddings'].tolist())

/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


(873, 21)
(873, 1)


In [4]:
#Exercise 1
threshold=.8

adj = (sim_matrix > threshold).astype(np.uint8)
np.fill_diagonal(adj, 0)

adj_sparse = csr_matrix(adj)

n_clusters, cluster_labels = connected_components(
    csgraph=adj_sparse,
    directed=False,

)
TEST_SIZE   = 0.2
RANDOM_SEED = 42
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_SEED)
train_idx_c, test_idx_c = next(gss.split(x, y, groups=cluster_labels))